# Strands Agents 4 - Multi-Agent Flows in Depth

A close look at *how teams of agents actually behave*: every way context moves between them, every failure point, what Strands does on each failure, how to stop failures, how to recover, and how to make the collaboration better.

This is the deep-dive companion to Notebook 3. It assumes you have seen Agents-as-Tools, Swarm, and Graph at least once. Everything here is checked against `strands-agents` 1.42 - including how node failures propagate, the exact handoff tool signature, and the hook payloads.

> **Cost note:** the orchestration demos make several model calls each. We default to **Haiku 4.5** to keep them fast and cheap. Cells that call Bedrock are marked `# >>> calls Bedrock`.

## 0 · Setup (run once)

Same one-time setup as the other notebooks: install, set region + credentials, pick a model, smoke-test.

In [ ]:
%pip install -q strands-agents strands-agents-tools
print("installed")

In [ ]:
import os
os.environ["AWS_DEFAULT_REGION"] = "us-west-2"   # <-- your enabled-model region
try:
    from google.colab import userdata
    os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID")
    os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
    print("Colab: credentials loaded from Secrets.")
except ModuleNotFoundError:
    print("Local: using your existing AWS credentials.")
print("Region:", os.environ["AWS_DEFAULT_REGION"])

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel
from botocore.config import Config

MODEL_HAIKU  = "us.anthropic.claude-haiku-4-5-20251001-v1:0"   # fast + cheap (default)
MODEL_SONNET = "us.anthropic.claude-sonnet-4-20250514-v1:0"    # stronger reasoning

def get_model(model_id=MODEL_HAIKU, temperature=0.3, max_tokens=2048):
    return BedrockModel(
        model_id=model_id, temperature=temperature, max_tokens=max_tokens,
        boto_client_config=Config(read_timeout=900, connect_timeout=900,
                                  retries={"max_attempts": 4, "mode": "adaptive"}),
    )

MODEL = get_model()

def say(result):
    """Flatten an AgentResult's reply to plain text."""
    return "".join(b.get("text", "") for b in result.message["content"]).strip()

def node_text(container, node_id):
    """Safely read one node's text from a Graph/Swarm RESULT or a GraphState.

    container.results[node_id] -> NodeResult ; .result -> AgentResult (or an Exception if it failed).
    Returns None if the node has not run or it failed.
    """
    nr = container.results.get(node_id)
    if nr is None or isinstance(nr.result, Exception):
        return None
    return "".join(b.get("text", "") for b in nr.result.message["content"]).strip()

print("ready:", MODEL.get_config()["model_id"])

In [ ]:
# >>> calls Bedrock  (confirms access + creds)
smoke = Agent(model=MODEL, callback_handler=None)
print(say(smoke("Reply with exactly: setup OK")))

---
## 1 · A mental model: who decides, and how context flows

Every multi-agent design answers two questions. Hold these in your head; the rest of the notebook is just the details.

**(a) Who decides the path?**

```
  YOU decide  ─────────────────────────────────►  the AGENTS decide
  (deterministic)                                  (emergent)

  Graph            Workflow         Agents-as-Tools         Swarm
  fixed edges      task deps        orchestrator routes     agents hand off
```

**(b) How does data move between agents?** Four distinct channels, covered next. Picking the wrong channel is the most common source of "the second agent didn't see what the first one found."

**In-process vs networked.** Agents-as-Tools, Graph, Swarm, and Workflow all run inside one Python process and share objects directly. **A2A** is the only one that crosses a process or machine boundary (JSON-RPC over HTTP); it trades simplicity for independent deployment.

Rule of thumb: **start deterministic** (Graph or Agents-as-Tools). Reach for a Swarm only when you genuinely cannot predict the order of work. Determinism is easier to test, cost, and debug.

---
## 2 · Context passing - the four channels

This is the crux. There are exactly four ways data reaches another agent, and they differ on one thing that matters a lot: **does the model see it?** (Anything the model sees costs tokens and can leak.)

| Channel | Who writes it | Model sees it? | Mutable mid-run? | Use it for |
|---|---|---|---|---|
| **1. Tool request/response** (Agents-as-Tools) | you, in the orchestrator | yes (it's the prompt/answer) | n/a | task instructions, results to reason over |
| **2. Dependency output** (Graph) | the framework, from each node's output | yes (becomes next node's input) | no | passing a stage's result to the next stage |
| **3. Handoff + shared context** (Swarm) | the agents, via `handoff_to_agent` | yes (next agent reads it) | yes (accumulates) | facts a teammate needs to continue |
| **4. `invocation_state`** (any pattern) | you, at call time | **no** | no | ids, DB handles, secrets, config |

Channels 1-3 are *model-visible* - that is their job. Channel 4 is the escape hatch for everything the model should **not** see.

### 2.1 · Channel 1 - Agents-as-Tools: a string in, a string out

You wrap a specialist in `@tool`. The orchestrator calls it with a **request string** and gets back the specialist's **text**. You compose exactly what the specialist sees, so context passing is fully under your control.

**Key nuance - isolation:** the orchestrator does **not** see the specialist's internal steps or tool calls. It only sees the returned string. Each specialist has its own private message history.

In [ ]:
# >>> calls Bedrock
@tool
def stats_expert(question: str) -> str:
    """Answer a statistics question with a one-line result."""
    # The orchestrator decides what 'question' contains - that IS the context handoff.
    sub = Agent(model=MODEL, system_prompt="You are a statistician. Answer in one line.",
                callback_handler=None)
    return say(sub(question))

orch = Agent(model=MODEL,
             system_prompt="Route math/stats questions to stats_expert, then summarize its answer for a beginner.",
             tools=[stats_expert], callback_handler=None)
print(say(orch("What does a p-value of 0.03 mean?")))

### 2.2 · Channel 2 - Graph: outputs flow along the edges

In a Graph you never pass data by hand. Strands builds each node's input from the outputs of the nodes that point into it (it reads them from `GraphState.results`). You shape *what* flows by choosing edges and writing node prompts.

After a run, you can see exactly what happened: `result.execution_order`, and each node's text via our `node_text` helper.

In [ ]:
# >>> calls Bedrock
from strands.multiagent import GraphBuilder

extract = Agent(name="extract", model=MODEL,
    system_prompt="Extract the 3 key facts from the input as bullets.", callback_handler=None)
rewrite = Agent(name="rewrite", model=MODEL,
    system_prompt="Rewrite the facts you receive as one tweet-length sentence.", callback_handler=None)

gb = GraphBuilder()
gb.add_node(extract, "extract")
gb.add_node(rewrite, "rewrite")
gb.add_edge("extract", "rewrite")          # extract's output becomes rewrite's input
gb.set_entry_point("extract")
g = gb.build()

res = g("The Eiffel Tower opened in 1889, is 330m tall, and was meant to be temporary.")
print("execution order:", [n.node_id for n in res.execution_order])
print("\nextract node said:\n", node_text(res, "extract"))
print("\nfinal (rewrite) node said:\n", node_text(res, "rewrite"))

### 2.3 · Channel 3 - Swarm: `handoff_to_agent` + shared context

When you build a `Swarm`, Strands injects exactly one coordination tool into every agent:

```python
handoff_to_agent(agent_name: str, message: str, context: dict | None = None)
```

- **`message`** is the note the next agent reads first.
- **`context`** (optional dict) is merged into a **shared context** that travels with the whole team. It is namespaced per sending agent and must be **JSON-serializable** (str, int, float, bool, list, dict, None) - hand off plain data, not Python objects.
- The receiving agent sees the message, the original task, and all accumulated shared context.

**Completion:** the swarm ends when an agent finishes **without** calling `handoff_to_agent` (or when a limit is hit). There is no separate "done" tool in this version - *not handing off* is the signal.

After a run, `result.node_history` shows the order agents actually ran in.

In [ ]:
# >>> calls Bedrock
from strands.multiagent import Swarm

triage = Agent(name="triage", model=MODEL,
    description="Classifies a request and routes it.",
    system_prompt=("Decide if a message is a REFUND or a TECH issue. "
                   "Then hand off to the matching agent using handoff_to_agent, "
                   "passing a short 'message'. Do not answer it yourself."),
    callback_handler=None)
refunds = Agent(name="refunds", model=MODEL,
    description="Handles refund requests.",
    system_prompt="Resolve refund requests politely in 2 sentences. Then finish (do not hand off).",
    callback_handler=None)
tech = Agent(name="tech", model=MODEL,
    description="Handles technical issues.",
    system_prompt="Give one concrete troubleshooting step. Then finish (do not hand off).",
    callback_handler=None)

swarm = Swarm([triage, refunds, tech], entry_point=triage, max_handoffs=4)
out = swarm("My app keeps crashing on startup after the latest update.")
print("agents that ran:", [n.node_id for n in out.node_history])
# the last agent that ran produced the answer:
last = out.node_history[-1].node_id
print("\nfinal answer from", last, "->\n", node_text(out, last))

### 2.4 · Channel 4 - `invocation_state`: data the model never sees

Pass a dict as `invocation_state=` to `agent(...)`, `graph(...)`, or `swarm(...)`. It reaches your **tools** (via `tool_context.invocation_state`) and nodes, but it is **never** sent to the model. This is where ids, database handles, API clients, and secrets belong - out of the prompt.

A tool reads it through a parameter named exactly **`tool_context`** (Strands injects it; the model does not see that parameter).

In [ ]:
# >>> calls Bedrock
@tool
def lookup_account(field: str, tool_context) -> str:
    """Look up a field ('tier' or 'region') for the current user."""
    inv = tool_context.invocation_state or {}
    user = inv.get("user_profile", {})
    return str(user.get(field, "unknown"))

acct_agent = Agent(model=MODEL, tools=[lookup_account], callback_handler=None)

# the profile rides in invocation_state - it is NOT in the prompt, so the model can't see or echo it
result = acct_agent(
    "What tier is this user, and which region are they in?",
    invocation_state={"user_profile": {"tier": "gold", "region": "ap-south-1"}},
)
print(say(result))

**Pain point to avoid:** stuffing large blobs or secrets into channels 1-3. A 5,000-word document pasted into a handoff message is sent to the model on every step (cost + latency), and a secret placed there can be echoed back. Put bulk/sensitive data in `invocation_state` (or `agent.state`) and pass only a reference through the model-visible channels.

---
## 3 · Pattern nuances that bite people

### 3.1 · Agents-as-Tools
- **The orchestrator prompt is the router.** Be explicit: "use X for ..., use Y for ..., then combine." Vague prompts cause skipped or doubled tool calls.
- **Right-size each specialist's model.** A cheap model for narrow specialists, a stronger one only for synthesis. You set the model per sub-agent.
- **Histories are isolated.** A specialist does not remember a previous specialist's call. If two specialists need shared facts, the orchestrator must pass them in the request string.

### 3.2 · Swarm
- **Every agent needs `name` and `description`.** Teammates choose who to hand to based on the description - write it like a job title.
- **Do not name your own tool `handoff_to_agent`.** Strands injects a tool by that exact name and raises a `ValueError` on a conflict. Rename yours.
- **Two opposite failure modes:** an agent that never hands off (ends too early) and agents that hand off forever (ping-pong). The next cell shows how to bound both.

In [ ]:
# Build a swarm with the safety knobs that prevent runaway / ping-pong handoffs.
# (Construction only - no Bedrock call here. The knobs are what matter.)
from strands.multiagent import Swarm

a = Agent(name="a", description="step A", model=MODEL, callback_handler=None)
b = Agent(name="b", description="step B", model=MODEL, callback_handler=None)

bounded = Swarm(
    [a, b],
    entry_point=a,
    max_handoffs=6,            # hard cap on total handoffs
    max_iterations=6,          # hard cap on total agent turns
    execution_timeout=120.0,   # wall-clock budget for the whole swarm (seconds)
    node_timeout=30.0,         # budget per single agent turn (seconds)
    repetitive_handoff_detection_window=4,   # look at the last 4 turns...
    repetitive_handoff_min_unique_agents=2,  # ...and stop if fewer than 2 unique agents (ping-pong)
)
print("swarm built with guardrails; nodes:", list(bounded.nodes))

### 3.3 · Graph - conditional edges give you branching

A plain Graph is a straight pipeline. The power feature is the **conditional edge**: `add_edge(from, to, condition=fn)` where `fn` takes the live `GraphState` and returns `True`/`False`. The edge is only traversed when the condition holds - that is how you build decision trees and routers with deterministic wiring.

Below: a `classify` node labels a message, and two conditional edges send it to the right specialist. The condition reads the classifier's text out of `GraphState` (using our `node_text` helper, which also works on a state object).

In [ ]:
# >>> calls Bedrock
from strands.multiagent import GraphBuilder

classify = Agent(name="classify", model=MODEL,
    system_prompt="Reply with exactly one word: BILLING or TECHNICAL.", callback_handler=None)
billing = Agent(name="billing", model=MODEL,
    system_prompt="Answer the billing question in 2 sentences.", callback_handler=None)
technical = Agent(name="technical", model=MODEL,
    system_prompt="Give one technical troubleshooting step.", callback_handler=None)

def route_to(label):
    # condition gets the GraphState; we read what 'classify' produced
    return lambda state: label in (node_text(state, "classify") or "").upper()

gb = GraphBuilder()
gb.add_node(classify, "classify")
gb.add_node(billing, "billing")
gb.add_node(technical, "technical")
gb.add_edge("classify", "billing",   condition=route_to("BILLING"))     # only if classified BILLING
gb.add_edge("classify", "technical", condition=route_to("TECHNICAL"))   # only if classified TECHNICAL
gb.set_entry_point("classify")
gb.set_max_node_executions(10)        # safety cap (graphs can be built with cycles)
router = gb.build()

r = router("I was charged twice this month.")
print("route taken:", [n.node_id for n in r.execution_order])
chosen = [n.node_id for n in r.execution_order if n.node_id in ("billing", "technical")]
print("answer:\n", node_text(r, chosen[-1]) if chosen else "(no branch matched)")

**More Graph nuances:**
- **Parallel siblings:** if two nodes both depend only on node X, they run **concurrently** after X. Good for speed; make sure they do not need each other's output.
- **Multiple entry points:** call `set_entry_point` more than once to start several independent branches at the top.
- **Loops:** edges can point backward. To loop safely, enable `reset_on_revisit(True)` (so a revisited node starts fresh) **and** set `set_max_node_executions(n)` so it cannot spin forever. We use exactly this for the self-correction loop in section 5.

---
## 4 · Failure points - and exactly what Strands does

This is the part people guess at. Here is the verified behavior, by layer. The single most important distinction:

> **Tool failures are contained. Node failures are fail-fast.**

| Failure | How Strands surfaces it | Prevent | Recover |
|---|---|---|---|
| A `@tool` raises | Framework catches it and returns a tool result with `status="error"` and the error text **to the model**. The agent keeps going and can react. | `try/except` in the tool to return a *clear, actionable* message; validate inputs | Let the model retry/ask; or set `AfterToolCallEvent.retry=True` in a hook |
| Model throttled / transient 5xx | Raised as a model error; the `adaptive` retry config retries with backoff | Keep `retries={"mode":"adaptive"}` (we do); reduce tokens; space calls | `AfterModelCallEvent` exposes `.exception` and `.retry` for custom retry |
| Context window overflow | `ContextWindowOverflowException` | Attach a `SlidingWindow...` or `Summarizing...` conversation manager | Switch/After the manager trims; shorten inputs |
| A node throws (uncaught) **or** hits `node_timeout` | **Fail-fast:** the exception propagates **out of** `graph(...)` / `swarm(...)`; overall `status = FAILED`; the failed node is recorded in `results[node].status = FAILED` with the `Exception` stored in `.result` | Make tools defensive; bound work with `node_timeout` | Wrap the call in `try/except`; inspect `failed_nodes`; fall back to a simpler path |
| Orchestration limit hit (`max_handoffs`, `max_iterations`, `execution_timeout`, repetitive handoff) | **Graceful stop** with a reason; status reflects it - **no crash** | Set the limits deliberately | Read the result; decide whether partial output is usable |
| Wrong routing / hallucinated handoff target | No error - just a bad path | Crisp `description`s, explicit orchestrator instructions | Add a validator node (section 5) that rejects bad output |

Let's see the two extremes for real.

In [ ]:
# Tool-failure containment: a tool that RAISES does not crash the agent.
# (No Bedrock - we call the tool directly to see what the model would receive.)
@tool
def risky_divide(x: int) -> int:
    """Divide 100 by x."""
    return 100 // x   # ZeroDivisionError when x == 0

ag = Agent(tools=[risky_divide])
print(ag.tool.risky_divide(x=0))   # -> a dict with status='error' and the message, NOT an exception

See it: `status='error'` with the error text. In a real run the **model receives this** as the tool's result and can apologize, try a different value, or ask the user. The agent loop did not crash. That is why tool-level problems rarely need orchestration-level recovery - but a *node* throwing is different, as the table says: it propagates out and you must catch it.

---
## 5 · Recovery patterns (hands-on)

### 5.1 · Defensive tools - turn failures into instructions

The framework wraps a raised exception as a generic `status="error"`. You can do better: catch it yourself and return a message the model can **act on** (what went wrong, what to try). This is the cheapest, highest-leverage reliability move.

In [ ]:
# No Bedrock - direct call shows the friendly, actionable result.
KNOWN = {"r1": "Alice", "r2": "Bob"}

@tool
def fetch_record(record_id: str) -> str:
    """Fetch a customer record by id."""
    try:
        if record_id not in KNOWN:
            return (f"ERROR: no record '{record_id}'. "
                    f"Known ids are {list(KNOWN)}. Ask the user to confirm the id.")
        return KNOWN[record_id]
    except Exception as e:                       # last-resort guard
        return f"ERROR while fetching '{record_id}': {type(e).__name__}: {e}"

ag = Agent(tools=[fetch_record])
print(ag.tool.fetch_record(record_id="r9"))      # actionable message, not a stack trace

### 5.2 · An observability + retry hook

A `HookProvider` lets you watch and intervene at lifecycle points without touching agent logic. The events you will use most:

- `AfterToolCallEvent` - has `.tool_use`, `.result`, `.exception`, and a writable `.retry` (set it to retry the tool).
- `AfterModelCallEvent` - has `.exception` and `.retry` (retry a transient model failure).
- `BeforeNodeCallEvent` / `AfterNodeCallEvent` - trace multi-agent node execution (`.node_id`).

You attach a provider with `Agent(hooks=[...])`, `Swarm(hooks=[...])`, or `GraphBuilder().set_hook_providers([...])`.

In [ ]:
# Construct + attach a tracer; it fires on a direct tool call (no Bedrock needed).
from strands.hooks import HookProvider, HookRegistry, AfterToolCallEvent, AfterModelCallEvent, BeforeNodeCallEvent

class Tracer(HookProvider):
    def __init__(self):
        self.events = []
    def register_hooks(self, registry: HookRegistry, **kwargs):
        registry.add_callback(AfterToolCallEvent, self.on_tool)
        registry.add_callback(AfterModelCallEvent, self.on_model)
        registry.add_callback(BeforeNodeCallEvent, self.on_node)
    def on_tool(self, e: AfterToolCallEvent):
        status = "error" if e.exception else "ok"
        self.events.append(f"tool {e.tool_use.get('name')} -> {status}")
        # Example recovery hook: retry once on a transient tool error
        # if e.exception and not getattr(e, '_retried', False):
        #     e._retried = True; e.retry = True
    def on_model(self, e: AfterModelCallEvent):
        if e.exception:
            self.events.append(f"model error: {type(e.exception).__name__}")
            # e.retry = True   # uncomment to auto-retry a transient model failure
    def on_node(self, e: BeforeNodeCallEvent):
        self.events.append(f"node start: {e.node_id}")

tracer = Tracer()

@tool
def greet(name: str) -> str:
    "Say hello."
    return f"Hello, {name}!"

ag = Agent(tools=[greet], hooks=[tracer])
ag.tool.greet(name="Akash")
print("captured events:", tracer.events)

### 5.3 · Wrap the orchestrator call + fall back

Because a node throwing is **fail-fast**, production code wraps `graph(...)` / `swarm(...)` in `try/except`. On failure you can inspect which node died and fall back to a single dependable agent so the user still gets an answer.

In [ ]:
# >>> calls Bedrock
from strands.multiagent import GraphBuilder
from strands.multiagent.base import Status

def run_with_fallback(task: str):
    try:
        step1 = Agent(name="step1", model=MODEL,
                      system_prompt="Summarize the request in one line.", callback_handler=None)
        step2 = Agent(name="step2", model=MODEL,
                      system_prompt="Answer the summarized request.", callback_handler=None)
        gb = GraphBuilder()
        gb.add_node(step1, "step1"); gb.add_node(step2, "step2")
        gb.add_edge("step1", "step2"); gb.set_entry_point("step1")
        gb.set_node_timeout(60); gb.set_max_node_executions(10)
        graph = gb.build()

        result = graph(task)
        if result.status == Status.FAILED:                       # completed-but-failed case
            print("graph failed nodes:", [n.node_id for n in result.failed_nodes])
            raise RuntimeError("graph reported FAILED")
        return node_text(result, "step2")

    except Exception as e:                                       # fail-fast exception OR our raise
        print(f"[fallback] multi-agent path failed ({type(e).__name__}): {e}")
        backup = Agent(model=MODEL, callback_handler=None)        # one robust agent
        return say(backup(task))

print(run_with_fallback("Give me one tip for learning to swim."))

### 5.4 · Self-correction loop (worker -> critic -> maybe loop)

The strongest reliability pattern: a **critic** node checks the worker's output and either approves (the graph stops) or sends it back for another pass. A conditional edge plus `reset_on_revisit(True)` plus `set_max_node_executions(n)` makes the loop **bounded** so it cannot run forever.

In [ ]:
# >>> calls Bedrock
from strands.multiagent import GraphBuilder

worker = Agent(name="worker", model=MODEL,
    system_prompt=("Write a haiku on the given topic. If you receive critic feedback, "
                   "revise to address it."), callback_handler=None)
critic = Agent(name="critic", model=MODEL,
    system_prompt=("Check the haiku is 3 lines and roughly 5-7-5. "
                   "Reply 'APPROVE' if good, otherwise reply 'REVISE' plus one specific fix."),
    callback_handler=None)

def needs_revision(state):
    txt = (node_text(state, "critic") or "").upper()
    return "REVISE" in txt        # loop back only while the critic says REVISE

gb = GraphBuilder()
gb.add_node(worker, "worker")
gb.add_node(critic, "critic")
gb.add_edge("worker", "critic")                       # worker output -> critic
gb.add_edge("critic", "worker", condition=needs_revision)  # critic feedback -> worker, only on REVISE
gb.set_entry_point("worker")
gb.reset_on_revisit(True)                             # fresh worker state each loop
gb.set_max_node_executions(6)                         # hard stop: at most 6 node runs
loop = gb.build()

r = loop("autumn rain")
print("execution order:", [n.node_id for n in r.execution_order])
print("\nfinal haiku:\n", node_text(r, "worker"))

### 5.5 · Circuit breakers (stop runaway cost)

Always cap an autonomous flow. The verified knobs:

- **Graph:** `set_node_timeout(seconds)`, `set_execution_timeout(seconds)`, `set_max_node_executions(n)`.
- **Swarm:** `node_timeout`, `execution_timeout`, `max_handoffs`, `max_iterations`, and `repetitive_handoff_detection_window` + `repetitive_handoff_min_unique_agents`.

Set these from day one. An agent that loops or hands off forever is a billing incident, not a bug you find later.

---
## 6 · Augmenting collaboration (making it genuinely better)

- **Right-size models per role.** Haiku for routers and narrow specialists, Sonnet only where deep reasoning pays off. You set `model=` per agent, so a 4-agent system can mix models and cut cost sharply.
- **Descriptions and prompts are the routing logic.** The model picks tools and handoff targets from `description`s. Tighten them and routing accuracy jumps - no code change.
- **Nest patterns instead of forcing one.** `GraphBuilder.add_node` accepts an `Agent` **or** a `MultiAgentBase`, so a whole `Swarm` or sub-`Graph` can be a single node in a larger Graph. Compose: deterministic outer Graph, emergent Swarm only inside the step that needs it.
- **Choose shared vs isolated memory deliberately.** Swarm shares context across the team (good for collaboration, more tokens). Agents-as-Tools keeps histories isolated (cheaper, more controlled). Pick per use case.
- **Instrument everything.** Each node result carries `accumulated_usage` (tokens) and you get `execution_order` (Graph) / `node_history` (Swarm). Add a tracing hook (5.2) for live visibility, and enable OpenTelemetry for production traces.
- **Use the determinism dial.** Prototype with a Swarm to discover the workflow, then, once you know the path, re-implement it as a Graph for repeatability, lower cost, and testability.

In [ ]:
# Per-node cost/observability after any run. Re-using `r` from the self-correction loop (5.4).
print("status        :", r.status)
print("total tokens  :", r.accumulated_usage["totalTokens"])
print("node count    :", r.total_nodes, "| completed:", len(r.completed_nodes), "| failed:", len(r.failed_nodes))
for n in r.execution_order:
    nr = r.results.get(n.node_id)
    toks = nr.accumulated_usage["totalTokens"] if nr else 0
    print(f"  {n.node_id:>8}: {toks} tokens")

---
## 7 · A consolidated, defensive flow

Putting the pieces together: a router Graph with a conditional branch, a **defensive tool**, a **tracing hook**, per-node **timeouts + execution cap**, and a **try/except fallback** around the whole thing. This is close to how you would ship a small multi-agent service.

In [ ]:
# >>> calls Bedrock
from strands.multiagent import GraphBuilder
from strands.multiagent.base import Status

# a defensive tool the specialists can use
ORDERS = {"1001": "shipped", "1002": "processing"}
@tool
def order_status(order_id: str) -> str:
    """Return the status of an order id."""
    if order_id not in ORDERS:
        return f"ERROR: unknown order '{order_id}'. Known: {list(ORDERS)}. Ask the user to re-check."
    return ORDERS[order_id]

def defensive_router(user_msg: str):
    try:
        classify = Agent(name="classify", model=MODEL,
            system_prompt="Reply with exactly one word: ORDER or GENERAL.", callback_handler=None)
        order_agent = Agent(name="order", model=MODEL, tools=[order_status],
            system_prompt="Use order_status to answer. If it returns an error, relay the guidance.",
            callback_handler=None)
        general = Agent(name="general", model=MODEL,
            system_prompt="Answer the general question in 2 sentences.", callback_handler=None)

        def to(label):
            return lambda s: label in (node_text(s, "classify") or "").upper()

        gb = GraphBuilder()
        gb.add_node(classify, "classify")
        gb.add_node(order_agent, "order")
        gb.add_node(general, "general")
        gb.add_edge("classify", "order",   condition=to("ORDER"))
        gb.add_edge("classify", "general", condition=to("GENERAL"))
        gb.set_entry_point("classify")
        gb.set_node_timeout(60)
        gb.set_execution_timeout(180)
        gb.set_max_node_executions(8)
        gb.set_hook_providers([Tracer()])          # the hook from 5.2
        graph = gb.build()

        res = graph(user_msg)
        if res.status == Status.FAILED:
            raise RuntimeError(f"graph FAILED at {[n.node_id for n in res.failed_nodes]}")
        leaf = [n.node_id for n in res.execution_order if n.node_id in ("order", "general")]
        return node_text(res, leaf[-1]) if leaf else "(no branch matched)"

    except Exception as e:
        print(f"[fallback] {type(e).__name__}: {e}")
        return say(Agent(model=MODEL, callback_handler=None)(user_msg))

print(defensive_router("What's the status of order 1002?"))
print("---")
print(defensive_router("What's the status of order 9999?"))   # defensive tool guides the model

---
## Recap + failure-prevention checklist

**The two facts to remember**
- **Tool exceptions are contained** (returned to the model as `status="error"`); **node exceptions are fail-fast** (they propagate out of `graph()`/`swarm()` and set `status=FAILED`).
- **Four context channels:** tool request/response, graph dependency outputs, swarm handoff + shared context (model-visible); `invocation_state` (hidden from the model - use for ids/secrets/handles).

**Before you ship a multi-agent flow**
- [ ] Every tool returns a clear message on bad input (defensive tools).
- [ ] The orchestrator call is wrapped in `try/except` with a single-agent fallback.
- [ ] `node_timeout` + `execution_timeout` + `max_node_executions` set (Graph); `max_handoffs`/`max_iterations`/timeouts + repetitive-handoff detection set (Swarm).
- [ ] Secrets/large data go through `invocation_state`, not the prompt.
- [ ] A tracing hook (or OpenTelemetry) is attached; you can see `execution_order` / `node_history` and per-node tokens.
- [ ] Conversation manager attached for long-running agents (overflow safety).
- [ ] Models right-sized per role; descriptions tight enough to route correctly.
- [ ] A validator/critic step guards quality where it matters, with a bounded loop.

Build it deterministic, bound every autonomous step, make tools defensive, and watch the path. That is reliable multi-agent Strands.